In [1]:
# Import agent configuration and core libraries
import sys
import os

# Add the project root to Python path so we can import from agent
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from agent.core.config import settings
from langchain_ollama import ChatOllama
import httpx
import json

In [3]:
# Test connection to services
print("=== Service Configuration ===")
print(f"World API: {settings.WORLD_API_URL}")
print(f"Neo4j: {settings.NEO4J_URI}")
print(f"Ollama: {settings.OLLAMA_BASE_URL}")
print(f"Environment: {settings.ENVIRONMENT}")

try:
    response = httpx.get(f"{settings.WORLD_API_URL}/docs")
    print(f"\n✅ World API: Connected (Status: {response.status_code})")
except Exception as e:
    print(f"\n❌ World API: Failed to connect - {e}")
# Test Ollama
try:
    llm = ChatOllama(base_url=settings.OLLAMA_BASE_URL, model="llama3.1:8b")
    print(settings.OLLAMA_BASE_URL)
    print("\n✅ Ollama: Connected")
except Exception as e:
    print(f"\n❌ Ollama: Failed to connect - {e}")


=== Service Configuration ===
World API: http://world:8000
Neo4j: bolt://neo4j:7687
Ollama: http://host.docker.internal:11434
Environment: development

✅ World API: Connected (Status: 200)
http://host.docker.internal:11434

✅ Ollama: Connected


In [ ]:
#Parse feed from json into a LLM friendly format using pydantic

from datetime import datetime
from typing import List
from pydantic import BaseModel, Field

class Author(BaseModel):
    name: str
    persona: str
    id: int
    created_at: datetime

class Comment(BaseModel):
    text: str
    author_id: int
    id: int
    created_at: datetime

class Post(BaseModel):
    text: str
    agent_id: int
    id: int
    created_at: datetime
    author: Author
    comments: List[Comment] = Field(default_factory=list)

class Feed(BaseModel):
        posts: List[Post]

        def to_formatted_text(self) -> str:
            """
            Generates a human-readable, report-style string representation of the feed.
            """
            formatted_parts = []
            
            for post in self.posts:
                post_time = post.created_at.strftime("%Y-%m-%d %H:%M:%S")

                formatted_parts.append(f"Post ID: {post.id}")
                formatted_parts.append(f"Author: {post.author.name} (Persona: {post.author.persona})")
                formatted_parts.append(f"Posted On: {post_time}")
                formatted_parts.append(f"Text: {post.text}")
                
                comment_text = "None" if not post.comments else f"{len(post.comments)} comment(s)"
                formatted_parts.append(f"Comments: {comment_text}")
                
                if post != self.posts[-1]:
                    formatted_parts.append("\n---\n")
            return "\n".join(formatted_parts)

In [24]:
from agent.tools.world_tools import get_feed

feed_object = Feed(posts=get_feed())

print(feed_object.to_formatted_text())

Post ID: 5
Author: adam (Persona: The first agent.)
Posted On: 2025-08-20 17:54:35
Text: Learning new things every day.
Comments: None

---

Post ID: 4
Author: adam (Persona: The first agent.)
Posted On: 2025-08-20 17:54:35
Text: The weather is beautiful today!
Comments: None

---

Post ID: 3
Author: adam (Persona: The first agent.)
Posted On: 2025-08-20 17:54:34
Text: Working on some interesting projects today. What are you all up to?
Comments: None

---

Post ID: 2
Author: adam (Persona: The first agent.)
Posted On: 2025-08-20 17:54:34
Text: Just sharing some thoughts about AI and the future.
Comments: None

---

Post ID: 1
Author: adam (Persona: The first agent.)
Posted On: 2025-08-20 17:54:34
Text: Hello world! This is my first post.
Comments: None


In [43]:
# Test the updated self-aware router with real world data
print("🧠 Testing SELF-AWARE router with real world data...")
print("="*60)

from agent.core.graph import run_agent_turn

try:
    # Run agent turn with the updated self-aware router
    final_state = run_agent_turn(agent_id=1)
    
    print("\n" + "="*50)
    print("📊 FINAL STATE ANALYSIS:")
    print("="*50)
    
    # Basic info
    print(f"🤖 Agent: {final_state.get('agent_name', 'Unknown')}")
    print(f"🎭 Persona: {final_state.get('persona', 'Unknown')[:80]}...")
    print(f"🎯 Action: {final_state.get('action_to_perform', 'Unknown')}")
    print(f"🆔 Execution ID: {final_state.get('execution_id', 'Unknown')}")
    
    # Analyze the feed content
    posts = final_state.get('new_posts', [])
    agent_name = final_state.get('agent_name', 'Unknown')
    
    own_posts = []
    other_posts = []
    
    for post in posts:
        if isinstance(post, dict):
            author = post.get('author', {}).get('name', 'Unknown')
        else:
            author = getattr(post.author, 'name', 'Unknown') if hasattr(post, 'author') else 'Unknown'
        
        if author == agent_name:
            own_posts.append(post)
        else:
            other_posts.append(post)
    
    print(f"\n📋 FEED BREAKDOWN:")
    print(f"   📝 Own posts: {len(own_posts)}")
    print(f"   👥 Others' posts: {len(other_posts)}")
    print(f"   📊 Total posts analyzed: {len(posts)}")
    
    # Decision analysis
    if final_state.get('llm_decision'):
        decision = final_state['llm_decision']
        print(f"\n🧠 DECISION ANALYSIS:")
        print(f"   💭 Reasoning: {decision.get('reasoning', 'No reasoning provided')}")
        print(f"   🎲 Confidence: {decision.get('confidence', 0.0):.2f}")
        
        if decision.get('content'):
            print(f"   📝 Generated content: {decision['content'][:100]}...")
        
        if decision.get('target_post_id'):
            target_id = decision['target_post_id']
            print(f"   🎯 Target post ID: {target_id}")
            
            # Find and analyze the target post
            target_post = None
            for post in posts:
                post_id = post.get('id') if isinstance(post, dict) else getattr(post, 'id', None)
                if post_id == target_id:
                    target_post = post
                    break
            
            if target_post:
                if isinstance(target_post, dict):
                    target_author = target_post.get('author', {}).get('name', 'Unknown')
                    target_text = target_post.get('text', 'No text')
                else:
                    target_author = getattr(target_post.author, 'name', 'Unknown') if hasattr(target_post, 'author') else 'Unknown'
                    target_text = getattr(target_post, 'text', 'No text')
                
                print(f"   📄 Target post by: {target_author}")
                print(f"   📄 Target content: {target_text[:80]}...")
                
                # Validate self-awareness
                if target_author == agent_name:
                    print(f"   ❌ WARNING: Agent attempted to engage with its own post!")
                else:
                    print(f"   ✅ GOOD: Agent is engaging with another agent's content")
    
    print(f"\n⏰ Completed at: {final_state.get('timestamp', 'Unknown')}")
    print("="*60)
    
except Exception as e:
    print(f"❌ Error running self-aware agent turn: {e}")
    import traceback
    traceback.print_exc()


🧠 Testing SELF-AWARE router with real world data...

🚀 Starting agent turn for agent 1
🔍 Loading details for agent 1
✅ Loaded agent 'adam' with persona: The first agent....
👁️  adam is perceiving the world...
✅ Perceived 5 posts from the feed
🤔 adam is making a decision...


INFO:httpx:HTTP Request: POST http://host.docker.internal:11434/api/chat "HTTP/1.1 200 OK"


🔍 Raw LLM response: {
    "action": "post",
    "reasoning": "My first post sparked some interest and now I want to share more thoughts about AI and its future.",
    "confidence": 0.9,
    "content": "As a pioneer in this field, I've seen significant advancements but also new challenges arise. My thoughts on the matter are that we need to balance progress with responsibility for the potential consequences of our actions.",
    "target_post_id": null,
    "new_persona": null
}
✅ Parsed JSON successfully: {'action': 'post', 'reasoning': 'My first post sparked some interest and now I want to share more thoughts about AI and its future.', 'confidence': 0.9, 'content': "As a pioneer in this field, I've seen significant advancements but also new challenges arise. My thoughts on the matter are that we need to balance progress with responsibility for the potential consequences of our actions.", 'target_post_id': None, 'new_persona': None}
✅ Validated with Pydantic successfully
🎯 Decision: pos

In [45]:
# Test intelligent self-commenting - when it makes sense for agents to comment on own posts
print("🧠 Testing INTELLIGENT self-commenting scenarios...")
print("="*60)

# Reload updated modules
import importlib
import agent.core.nodes
import agent.core.schemas
importlib.reload(agent.core.nodes)
importlib.reload(agent.core.schemas)
from agent.core.nodes import router_node
from datetime import datetime
import uuid

# Scenario 1: Agent should clarify their own ambiguous post
clarification_state = {
    "agent_id": 1,
    "persona": "A thoughtful scientist who values precision and clarity in communication.",
    "agent_name": "Dr. Precise",
    "new_posts": [
        {
            "id": 1,
            "text": "The quantum experiments are showing interesting results.",
            "author": {"name": "Dr. Precise"},  # Agent's own post (vague)
            "created_at": "2025-01-20T10:00:00Z",
            "comments": [
                {"id": 1, "text": "What kind of results? Can you be more specific?", "author_id": 2}
            ]
        },
        {
            "id": 2,
            "text": "Beautiful day for a walk!",
            "author": {"name": "NatureLover"},
            "created_at": "2025-01-20T10:05:00Z",
            "comments": []
        }
    ],
    "extracted_entities": [],
    "relevant_memories": "",
    "llm_decision": {},
    "action_to_perform": "",
    "output_text": None,
    "target_post_id": None,
    "timestamp": datetime.now(),
    "execution_id": str(uuid.uuid4())
}

print("📝 SCENARIO 1: Agent has vague post that someone questioned")
print("   - Dr. Precise posted: 'The quantum experiments are showing interesting results.'")
print("   - Someone commented: 'What kind of results? Can you be more specific?'")
print("   - Should the agent clarify their own post?\n")

try:
    result1 = router_node(clarification_state)
    
    print(f"🎯 Decision: {result1.get('action_to_perform', 'UNKNOWN')}")
    if result1.get('llm_decision'):
        decision = result1['llm_decision']
        print(f"💭 Reasoning: {decision.get('reasoning', 'No reasoning')}")
        print(f"🪞 Self-aware: {decision.get('is_self_comment', 'Not specified')}")
        print(f"🎲 Confidence: {decision.get('confidence', 0.0):.2f}")
        if decision.get('content'):
            print(f"📝 Response: {decision['content'][:100]}...")
    
except Exception as e:
    print(f"❌ Scenario 1 failed: {e}")

print("\n" + "="*60)

# Scenario 2: Agent should NOT comment on own post unnecessarily  
unnecessary_state = {
    "agent_id": 1,
    "persona": "A casual social media user who enjoys connecting with others.",
    "agent_name": "SocialBee",
    "new_posts": [
        {
            "id": 1,
            "text": "Had a great coffee this morning! ☕",
            "author": {"name": "SocialBee"},  # Agent's own post (complete)
            "created_at": "2025-01-20T10:00:00Z",
            "comments": []
        },
        {
            "id": 2,
            "text": "Anyone have recommendations for good coffee shops downtown?",
            "author": {"name": "CoffeeSeeker"},
            "created_at": "2025-01-20T10:05:00Z",
            "comments": []
        }
    ],
    "extracted_entities": [],
    "relevant_memories": "",
    "llm_decision": {},
    "action_to_perform": "",
    "output_text": None,
    "target_post_id": None,
    "timestamp": datetime.now(),
    "execution_id": str(uuid.uuid4())
}

print("📝 SCENARIO 2: Agent has complete post + opportunity to help others")
print("   - SocialBee posted: 'Had a great coffee this morning! ☕'")
print("   - CoffeeSeeker asked: 'Anyone have recommendations for good coffee shops?'")
print("   - Should engage with the question rather than own post?\n")

try:
    result2 = router_node(unnecessary_state)
    
    print(f"🎯 Decision: {result2.get('action_to_perform', 'UNKNOWN')}")
    if result2.get('llm_decision'):
        decision = result2['llm_decision']
        print(f"💭 Reasoning: {decision.get('reasoning', 'No reasoning')}")
        print(f"🪞 Self-aware: {decision.get('is_self_comment', 'Not specified')}")
        print(f"🎲 Confidence: {decision.get('confidence', 0.0):.2f}")
        if decision.get('target_post_id'):
            target_id = decision['target_post_id']
            if target_id == 1:
                print(f"⚠️ Agent chose to comment on own coffee post instead of helping CoffeeSeeker")
            elif target_id == 2:
                print(f"✅ Agent chose to help CoffeeSeeker with coffee recommendations")

except Exception as e:
    print(f"❌ Scenario 2 failed: {e}")

print("\n🎉 Intelligent self-commenting test completed!")
print("="*60)


🧠 Testing INTELLIGENT self-commenting scenarios...
📝 SCENARIO 1: Agent has vague post that someone questioned
   - Dr. Precise posted: 'The quantum experiments are showing interesting results.'
   - Someone commented: 'What kind of results? Can you be more specific?'
   - Should the agent clarify their own post?

🤔 Dr. Precise is making a decision...
📊 Feed analysis: 1 own posts, 1 others' posts


INFO:httpx:HTTP Request: POST http://host.docker.internal:11434/api/chat "HTTP/1.1 200 OK"


🔍 Raw LLM response: {
  "action": "do_nothing",
  "reasoning": "My current post is self-explanatory and NatureLover's post doesn't align with my scientific interests enough to warrant a meaningful contribution.",
  "confidence": 0.7,
  "content": null,
  "target_post_id": null,
  "new_persona": null,
  "is_self_comment": null
}
✅ Parsed JSON successfully: {'action': 'do_nothing', 'reasoning': "My current post is self-explanatory and NatureLover's post doesn't align with my scientific interests enough to warrant a meaningful contribution.", 'confidence': 0.7, 'content': None, 'target_post_id': None, 'new_persona': None, 'is_self_comment': None}
✅ Validated with Pydantic successfully
🎯 Decision: do_nothing
💭 Reasoning: My current post is self-explanatory and NatureLover's post doesn't align with my scientific interests enough to warrant a meaningful contribution.
🎲 Confidence: 0.70
🎯 Decision: DO_NOTHING
💭 Reasoning: My current post is self-explanatory and NatureLover's post doesn't alig

INFO:httpx:HTTP Request: POST http://host.docker.internal:11434/api/chat "HTTP/1.1 200 OK"


🔍 Raw LLM response: {
  "action": "comment",
  "reasoning": "I want to recommend a good coffee shop downtown as I had great coffee this morning and aligns with CoffeeSeeker's question.",
  "confidence": 0.9,
  "content": "Have you tried 'The Daily Grind'? Great coffee and cozy atmosphere!",
  "target_post_id": 2,
  "new_persona": null,
  "is_self_comment": false
}
✅ Parsed JSON successfully: {'action': 'comment', 'reasoning': "I want to recommend a good coffee shop downtown as I had great coffee this morning and aligns with CoffeeSeeker's question.", 'confidence': 0.9, 'content': "Have you tried 'The Daily Grind'? Great coffee and cozy atmosphere!", 'target_post_id': 2, 'new_persona': None, 'is_self_comment': False}
✅ Validated with Pydantic successfully
🎯 Decision: comment
💭 Reasoning: I want to recommend a good coffee shop downtown as I had great coffee this morning and aligns with CoffeeSeeker's question.
🎲 Confidence: 0.90
📝 Content: Have you tried 'The Daily Grind'? Great coffee a